# Devkota: train on Colab

Runtime → Change runtime type → **T4 GPU**.

Before running, upload `devkota_data.zip` (containing `data/merged/` and `data/preprocessed/`) to `MyDrive/devkota/`.
Checkpoints are saved to `MyDrive/devkota/checkpoints`, so they survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/devkota'
CKPT = f'{DRIVE}/checkpoints'

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + data
Data is unzipped into the repo root so `data/merged` and `data/preprocessed` sit where `main.py` expects them.

In [ ]:
%cd /content
!test -d devkota || git clone https://github.com/un1u3/devkota.git
%cd /content/devkota
!git pull
!pip install -q sentencepiece
!unzip -qo {DRIVE}/devkota_data.zip -d /content/devkota
!ls data/merged data/preprocessed
%cd /content/devkota/devkota-v0.7

## 2. Tokenize

In [ ]:
!python main.py prepare

## 3. Pretrain on the general Nepali corpus
If Colab disconnects: rerun sections 1–2, then the **resume** cell below.

In [ ]:
!python main.py pretrain --ckpt-dir {CKPT} \
    --d-model 384 --num-layers 6 --num-heads 6 --max-seq-len 256 \
    --batch-size 32 --steps 8000 --eval-every 500 --lr 3e-4

In [ ]:
# resume pretraining after a disconnect (same settings as above)
!python main.py pretrain --ckpt-dir {CKPT} --resume \
    --batch-size 32 --steps 8000 --eval-every 500 --lr 3e-4

## 4. Fine-tune on the poems
Starts from `CKPT/pretrain.pt` (the best pretraining checkpoint).

In [ ]:
!python main.py finetune --ckpt-dir {CKPT} \
    --steps 1000 --eval-every 100 --lr 1e-4

## 5. Generate

In [ ]:
!python main.py generate --ckpt-dir {CKPT} --prompt "हिमाल" --max-tokens 200 --temperature 0.8